## **SHAP Analysis**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
import shap
import xgboost as xgb
from scipy.special import logit

In [ ]:
RUN_ID = "cd012efb051d43439053e27deca87e38"

TARGET_COL   = "REBOUND_TYPE"
ID_COLS = ["GAME_ID", "EVENT_NUM", "DETECTION_CONFIDENCE", "SHOT_DETECTION_CONFIDENCE", "RIM_DETECTION_CONFIDENCE"]
MLFLOW_DIR   = os.path.abspath("../../runs/mlruns")
HOLDOUT_PATH = "../../data/sequences/{moment}/features/tabular/sequences_fe_tabular_holdout.csv"

TOP_N        = 20

#### **Load Run & Data**

In [ ]:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"   # file-based store is in maintenance mode; allow reading it

mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR.replace(os.sep, '/')}")
run    = mlflow.tracking.MlflowClient().get_run(RUN_ID)
params = run.data.params
MOMENT = params["moment"]

pipe       = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/model")
est        = pipe.named_steps["model"]
MODEL_NAME = type(est).__name__

OUT_DIR = os.path.abspath(f"../../runs/shap/{MOMENT}_{MODEL_NAME}_{RUN_ID[:8]}")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{MOMENT} | {MODEL_NAME} | commit {params.get('git_commit', '?')}\n→ {OUT_DIR}")

In [ ]:
df = pd.read_csv(HOLDOUT_PATH.format(moment=MOMENT))
FEATURE_COLS = list(getattr(pipe, "feature_names_in_",
                            [c for c in df.columns if c != TARGET_COL and c not in ID_COLS]))
assert len(FEATURE_COLS) == pipe.n_features_in_, "feature count mismatch between CSV and model"

def to_in(X):
    return X if hasattr(pipe, "feature_names_in_") else np.asarray(X)

X_raw    = df[FEATURE_COLS].astype(float)
y        = df[TARGET_COL].values
X_scaled = pipe.named_steps["scale"].transform(to_in(X_raw))
proba    = pipe.predict_proba(to_in(X_raw))[:, 1]
print(f"Holdout: {len(df)} events | {len(FEATURE_COLS)} features | ORB rate {y.mean():.3f}")

#### **SHAP Values**

In [ ]:
def explain():
    all_idx = np.arange(len(X_raw))

    if MODEL_NAME == "XGBClassifier":
        booster = est.get_booster()
        dm = xgb.DMatrix(X_scaled, feature_names=booster.feature_names)
        c = booster.predict(dm, pred_contribs=True)
        return c[:, :-1], c[:, -1], all_idx, "log-odds"

    if MODEL_NAME in ("LGBMClassifier", "RandomForestClassifier"):
        e = shap.TreeExplainer(est)(X_scaled)
        v, b = e.values, e.base_values
        if v.ndim == 3:
            v, b = v[:, :, 1], b[:, 1]
        return v, b, all_idx, ("probability" if MODEL_NAME == "RandomForestClassifier" else "log-odds")

    if MODEL_NAME == "LogisticRegression":
        e = shap.LinearExplainer(est, X_scaled)(X_scaled)
        return e.values, e.base_values, all_idx, "log-odds"

    raise ValueError(f"unsupported model for SHAP: {MODEL_NAME}")


values, base, idx, SPACE = explain()
expl = shap.Explanation(values=values, base_values=base, data=X_raw.values[idx], feature_names=FEATURE_COLS)

p_raw  = proba[idx]
target = p_raw if SPACE == "probability" else logit(np.clip(p_raw, 1e-6, 1 - 1e-6))
print(f"Explained {len(idx)} events in {SPACE} space | "
      f"additivity max err: {np.abs(values.sum(1) + base - target).max():.2e}")

#### **Global Importance**

In [ ]:
FEATURE_CATEGORIES = {
    'Shot':             ['SHOT_DISTANCE','SHOT_X','SHOT_Y','SHOT_ZONE','SHOT_TYPE',
                         'SHOT_ANGLE','BALL_HEIGHT','SHOT_CLOCK','BALL_VX','BALL_VY',
                         'BALL_VZ','SHOT_CLOCK_BUCKET','SHOOTER_DIST_TO_BASKET'],
    'Paint Occupation': ['VORONOI_OFF_PAINT_PCT','PAINT_TOTAL_OFF','PAINT_DIFF',
                         'PAINT_AVG_TRB_OFF','PAINT_TRB_OFF','PAINT_TRB_DEF',
                         'PAINT_OFF_DEF_PROXIMITY','CONTESTED_PAINT_OFF',
                         'OFF_PLAYERS_IN_PAINT_RATIO'],
    '3pt Occupation':   ['VORONOI_OFF_3PT_PCT','3PT_TOTAL_OFF','3PT_DIFF'],
    'Box-out':          ['TOTAL_NOT_BOXED_OUT_PAINT','TOTAL_GREAT_ORB_POSITION',
                         'TOTAL_OPEN_PAINT','TOTAL_OPEN'],
    'Positioning':      ['CLOSEST_BASKET_TEAM','CLOSEST_OFF_DIST','2ND_CLOSEST_OFF_DIST',
                         '3RD_CLOSEST_OFF_DIST','OFF_PLAYERS_IN_TOP2_CLOSEST_BASKET',
                         'OFF_PLAYERS_IN_TOP3_CLOSEST_BASKET','OFF_PLAYERS_IN_TOP4_CLOSEST_BASKET',
                         'OFF_PLAYERS_IN_TOP5_CLOSEST_BASKET','CLOSEST_DEF_DIST',
                         '2ND_CLOSEST_DEF_DIST','AVG_OFF_DIST','AVG_DEF_DIST',
                         'OFF_PLAYERS_BEHIND_BASKET','DEF_IMBALANCE','OFF_DEF_DIST_DIFF',
                         'RIM_PROXIMITY_ADVANTAGE','CLOSEST_DEF_TO_CLOSEST_OFF',
                         'CLOSEST_DEF_TO_2ND_OFF','CLOSEST_OFF_ANGLE_TO_BASKET'],
    'Spacing':          ['OFF_SPREAD_STD','OFF_SPREAD_HULL','DEF_SPREAD_STD',
                         'DEF_SPREAD_HULL','OFF_CLUSTER_DIST','SPREAD_RATIO'],
    'Rebound Approach': ['AVG_VBASKET_OFF','AVG_VBASKET_DEF','MAX_VBASKET_OFF',
                         'SHOOTER_VBASKET','OFF_ATTACKING','PCT_OFF_ATTACKING',
                         'CLOSEST_DEF_RECOVERY_ANGLE','2ND_CLOSEST_DEF_RECOVERY_ANGLE'],
    'Rebound Ability':  ['ORB_DRB_MATCHUP_DIFF','CLOSEST_OFF_ORB','2ND_CLOSEST_OFF_ORB',
                         'CLOSEST_DEF_DRB','2ND_CLOSEST_DEF_DRB',
                         'CLOSEST_OFF_HEIGHT','CLOSEST_DEF_HEIGHT','CLOSEST_HEIGHT_DIFF'],
    'Landing Zone':     ['PRED_LANDING_X','PRED_LANDING_Y','CLOSEST_LANDING_ZONE_TEAM',
                         'CLOSEST_OFF_LANDING_ZONE_DIST','CLOSEST_DEF_LANDING_ZONE_DIST',
                         'CLOSEST_LANDING_ZONE_DIST_DIFF','OFF_NEAR_LANDING_ZONE',
                         'DEF_NEAR_LANDING_ZONE','CLOSEST_OFF_LANDING_ANGLE_TO_BASKET',
                         'CLOSEST_DEF_LANDING_ANGLE_TO_BASKET'],
}

def base_name(f):
    for p in ("SHOT_TYPE_", "SHOT_ZONE_"):
        if f.startswith(p):
            return p[:-1]
    for s in ("_SIN", "_COS"):
        if f.endswith(s):
            return f[:-len(s)]
    return f

feat_to_cat = {f: c for c, fs in FEATURE_CATEGORIES.items() for f in fs}
feat_to_cat = {f: feat_to_cat[base_name(f)] for f in FEATURE_COLS if base_name(f) in feat_to_cat}
unmapped = [f for f in FEATURE_COLS if f not in feat_to_cat]
if unmapped:
    raise ValueError(f"Uncategorized: {unmapped}")

print(pd.Series(feat_to_cat).value_counts())

In [ ]:
shap_df = pd.DataFrame(values, columns=FEATURE_COLS)
imp = pd.DataFrame({
    "mean_abs_shap": np.abs(values).mean(0),
    "mean_shap":     values.mean(0),
    "direction":     shap_df.corrwith(X_raw.iloc[idx].reset_index(drop=True), method="spearman"),
}, index=FEATURE_COLS)
imp["category"] = imp.index.map(feat_to_cat).fillna("Other")
imp = imp.sort_values("mean_abs_shap", ascending=False)
imp["rank"] = np.arange(1, len(imp) + 1)
imp.to_csv(os.path.join(OUT_DIR, "shap_importance.csv"))
imp.head(TOP_N).round(4)

In [ ]:
def save(name):
    plt.gcf().savefig(os.path.join(OUT_DIR, name), dpi=150, bbox_inches="tight")
    plt.show()

MOMENT_LABEL = {"moment_1": "Shot release", "moment_2": "Rim arrival"}

# restrict to the top 10 features so SHAP doesn't add a "Sum of N other features" row
N_PLOT   = 10
expl_top = expl[:, [FEATURE_COLS.index(f) for f in imp.index[:N_PLOT]]]

shap.plots.bar(expl_top, max_display=N_PLOT, show=False)
plt.title(f"{MOMENT_LABEL[MOMENT]} — top {N_PLOT} features by mean |SHAP| ({SPACE})")
save("bar.png")

In [ ]:
shap.plots.beeswarm(expl_top, max_display=N_PLOT, show=False)
plt.title(f"{MOMENT_LABEL[MOMENT]} — SHAP value distribution, top {N_PLOT} features ({SPACE})")
save("beeswarm.png")

In [ ]:
cat_imp = imp.groupby("category")["mean_abs_shap"].agg(["sum", "mean", "count"]).sort_values("sum")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(cat_imp.index, cat_imp["sum"])
ax.set_xlabel(f"Σ mean |SHAP| ({SPACE})")
ax.set_title(f"{MOMENT_LABEL[MOMENT]} — SHAP importance by feature category")
save("categories.png")
cat_imp.sort_values("sum", ascending=False).round(4)